In [ ]:
# SPR 2026 - mDeBERTa-v3-base + Focal Loss + Optuna Calibration (modelo proprio)
# mDeBERTa usa disentangled attention, superior a BERT em muitos benchmarks.
# Treino completo 5-fold CV com Focal Loss + Label Smoothing.

import os
import re
import gc
import numpy as np
import pandas as pd
from pathlib import Path

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.cuda.amp import autocast, GradScaler
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    get_linear_schedule_with_warmup,
)
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score
from tqdm.auto import tqdm
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)
import warnings
warnings.filterwarnings('ignore')

# ==========================================================================
# CONFIG
# ==========================================================================
SEED = 42
MAX_LEN = 512
BATCH_SIZE = 16  # mDeBERTa-base is smaller than BERTimbau Large
EPOCHS = 7  # more epochs since smaller model
LR = 2e-5
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.1
N_FOLDS = 5
NUM_CLASSES = 7
FOCAL_GAMMA = 2.0
FOCAL_ALPHA = 0.25
N_TRIALS = 300
LABEL_SMOOTHING = 0.05

MODEL_CANDIDATES = [
    '/kaggle/input/models/microsoft/mdeberta-v3-base/pytorch/default/1',
    '/kaggle/input/mdeberta-v3-base',
]
# Note: mDeBERTa does NOT have token_type_ids

DATA_DIR = '/kaggle/input/competitions/spr-2026-mammography-report-classification'

def find_model_path():
    """Tenta candidatos conhecidos, depois busca recursiva."""
    for c in MODEL_CANDIDATES:
        if os.path.isdir(c) and os.path.exists(os.path.join(c, 'config.json')):
            return c
    base = '/kaggle/input'
    def has_config(p):
        return os.path.isdir(p) and os.path.exists(os.path.join(p, 'config.json'))
    def search(d, depth=0):
        if depth > 10:
            return None
        try:
            for item in os.listdir(d):
                p = os.path.join(d, item)
                if os.path.isdir(p):
                    if has_config(p):
                        return p
                    r = search(p, depth + 1)
                    if r:
                        return r
        except Exception:
            pass
        return None
    return search(base)

MODEL_PATH = find_model_path()
if MODEL_PATH is None:
    raise FileNotFoundError('Modelo mDeBERTa nao encontrado. Adicione via Add Input.')

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

def seed_everything(seed):
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

seed_everything(SEED)

print(f'Device: {DEVICE}')
print(f'Model: {MODEL_PATH}')

In [ ]:
# ==========================================================================
# PREPROCESSING
# ==========================================================================
train_df = pd.read_csv(f'{DATA_DIR}/train.csv')
test_df = pd.read_csv(f'{DATA_DIR}/test.csv')
print(f'Train: {train_df.shape}, Test: {test_df.shape}')
print(f'Distribuicao:\n{train_df["target"].value_counts().sort_index()}')

INDICACAO_MARKERS = [
    'indicação clínica:\n', 'indicação clínica:',
    'indicação:', 'indicacao:',
]
ACHADOS_MARKERS = ['achados:\n', 'achados:']
COMPARATIVA_MARKERS = ['análise comparativa:', 'analise comparativa:']


def extract_section(text, start_markers, end_markers=None):
    txt_lower = text.lower()
    start_idx = -1
    for m in start_markers:
        idx = txt_lower.find(m)
        if idx >= 0:
            start_idx = idx + len(m)
            break
    if start_idx < 0:
        return ''
    if end_markers is None:
        return text[start_idx:].strip()
    end_idx = len(text)
    for m in end_markers:
        idx = txt_lower.find(m, start_idx)
        if 0 < idx < end_idx:
            end_idx = idx
    return text[start_idx:end_idx].strip()


def clean_text(text):
    text = re.sub(r'[\x00-\x08\x0b\x0c\x0e-\x1f\x7f\x81\x8d\x8f\x90\x9d\xad]', '', text)
    text = re.sub(r'[\n\r\t]+', ' ', text)
    text = re.sub(r' {2,}', ' ', text)
    return text.strip()


def build_input_text(report_text):
    report = clean_text(report_text)
    indicacao = extract_section(report, INDICACAO_MARKERS, ACHADOS_MARKERS)
    achados = extract_section(report, ACHADOS_MARKERS, COMPARATIVA_MARKERS)
    comparativa = extract_section(report, COMPARATIVA_MARKERS)
    if len(achados) < 10:
        return report
    parts = []
    if indicacao:
        parts.append(f'Indicação: {indicacao}')
    if achados:
        parts.append(f'Achados: {achados}')
    if comparativa:
        parts.append(f'Comparativa: {comparativa}')
    return ' '.join(parts)


train_df['text'] = train_df['report'].apply(build_input_text)
test_df['text'] = test_df['report'].apply(build_input_text)
print('Preprocessing concluido.')

In [ ]:
# ==========================================================================
# DATASET + FOCAL LOSS
# mDeBERTa NAO usa token_type_ids. Apenas input_ids e attention_mask.
# ==========================================================================
class MammographyDataset(Dataset):
    def __init__(self, texts, labels=None, tokenizer=None, max_len=512):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        enc = self.tokenizer(
            str(self.texts[idx]),
            truncation=True,
            max_length=self.max_len,
            padding='max_length',
            return_tensors='pt',
        )
        # mDeBERTa: only input_ids and attention_mask (NO token_type_ids)
        item = {
            'input_ids': enc['input_ids'].squeeze(0),
            'attention_mask': enc['attention_mask'].squeeze(0),
        }
        if self.labels is not None:
            item['labels'] = torch.tensor(self.labels[idx], dtype=torch.long)
        return item


class FocalLoss(nn.Module):
    def __init__(self, gamma=2.0, alpha=0.25, label_smoothing=0.0, num_classes=7):
        super().__init__()
        self.gamma = gamma
        self.alpha = alpha
        self.label_smoothing = label_smoothing
        self.num_classes = num_classes

    def forward(self, inputs, targets):
        if self.label_smoothing > 0:
            targets_smooth = torch.zeros_like(inputs).scatter_(
                1, targets.unsqueeze(1), 1.0
            )
            targets_smooth = (
                targets_smooth * (1 - self.label_smoothing)
                + self.label_smoothing / self.num_classes
            )
            ce_loss = -(targets_smooth * F.log_softmax(inputs, dim=-1)).sum(dim=-1)
        else:
            ce_loss = F.cross_entropy(inputs, targets, reduction='none')

        pt = torch.exp(-ce_loss)
        focal_loss = self.alpha * (1 - pt) ** self.gamma * ce_loss
        return focal_loss.mean()


tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH, local_files_only=True)
print(f'Tokenizer carregado: {tokenizer.__class__.__name__}')
print(f'Vocab size: {tokenizer.vocab_size}')

In [ ]:
# ==========================================================================
# TRAINING HELPERS (FP16 via GradScaler, mDeBERTa safe)
# ==========================================================================
def train_epoch(model, loader, criterion, optimizer, scheduler, scaler):
    model.train()
    total_loss = 0.0
    for batch in loader:
        input_ids = batch['input_ids'].to(DEVICE)
        attention_mask = batch['attention_mask'].to(DEVICE)
        labels = batch['labels'].to(DEVICE)

        optimizer.zero_grad()
        with autocast(dtype=torch.float16):
            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            loss = criterion(outputs.logits, labels)

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()
        total_loss += loss.item()

    return total_loss / len(loader)


@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    all_probs = []
    all_labels = []
    for batch in loader:
        input_ids = batch['input_ids'].to(DEVICE)
        attention_mask = batch['attention_mask'].to(DEVICE)
        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        probs = F.softmax(outputs.logits.float(), dim=-1).cpu().numpy()
        all_probs.append(probs)
        if 'labels' in batch:
            all_labels.extend(batch['labels'].numpy())
    all_probs = np.concatenate(all_probs, axis=0)
    if all_labels:
        all_labels = np.array(all_labels)
        preds = all_probs.argmax(axis=1)
        f1 = f1_score(all_labels, preds, average='macro')
        return all_probs, all_labels, f1
    return all_probs, None, None


def compute_f1(probs, labels):
    preds = probs.argmax(axis=1)
    return f1_score(labels, preds, average='macro')


print('Training helpers definidos.')

In [ ]:
# ==========================================================================
# CALIBRATION FUNCTIONS
# ==========================================================================
def temperature_scale(probs, temperature):
    logits = np.log(probs + 1e-10)
    scaled = logits / temperature
    exp_scaled = np.exp(scaled - scaled.max(axis=1, keepdims=True))
    return exp_scaled / exp_scaled.sum(axis=1, keepdims=True)


def apply_thresholds(probs, thresholds):
    adjusted = probs * np.array(thresholds)
    return adjusted.argmax(axis=1)


def optuna_joint_optimization(oof_probs, oof_labels, n_trials=300):
    """Otimiza temperatura + threshold multipliers conjuntamente."""
    def objective(trial):
        temp = trial.suggest_float('temperature', 0.3, 3.0)
        calibrated = temperature_scale(oof_probs, temp)
        thresholds = [
            trial.suggest_float(f't{c}', 0.05, 3.0) for c in range(NUM_CLASSES)
        ]
        preds = apply_thresholds(calibrated, thresholds)
        return f1_score(oof_labels, preds, average='macro')

    study = optuna.create_study(
        direction='maximize',
        sampler=optuna.samplers.TPESampler(seed=SEED),
    )
    study.optimize(objective, n_trials=n_trials, show_progress_bar=True)

    best_temp = study.best_params['temperature']
    best_thresholds = [study.best_params[f't{c}'] for c in range(NUM_CLASSES)]
    print(f'Melhor F1 OOF (Optuna): {study.best_value:.5f}')
    print(f'Temperatura: {best_temp:.4f}')
    print(f'Thresholds: {[round(t, 4) for t in best_thresholds]}')
    return best_temp, best_thresholds, study.best_value


print('Calibration functions definidas.')

In [ ]:
# ==========================================================================
# 5-FOLD TRAINING
# ==========================================================================
skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)

texts = train_df['text'].values
labels = train_df['target'].values

oof_probs = np.zeros((len(train_df), NUM_CLASSES))
oof_labels = np.zeros(len(train_df), dtype=int)
test_probs = np.zeros((len(test_df), NUM_CLASSES))
fold_f1s = []

test_ds = MammographyDataset(
    test_df['text'].values, None, tokenizer, MAX_LEN
)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

for fold, (train_idx, val_idx) in enumerate(skf.split(texts, labels)):
    print(f'\n{"="*60}')
    print(f'FOLD {fold + 1}/{N_FOLDS}')
    print(f'{"="*60}')

    seed_everything(SEED + fold)

    train_ds = MammographyDataset(
        texts[train_idx], labels[train_idx], tokenizer, MAX_LEN
    )
    val_ds = MammographyDataset(
        texts[val_idx], labels[val_idx], tokenizer, MAX_LEN
    )
    train_loader = DataLoader(
        train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=0
    )
    val_loader = DataLoader(
        val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0
    )

    # Carregar modelo do zero (pretrained, nao fine-tuned)
    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_PATH,
        num_labels=NUM_CLASSES,
        local_files_only=True,
    )
    model = model.float()  # mDeBERTa: forcar FP32 nos parametros base
    model.to(DEVICE)

    criterion = FocalLoss(
        gamma=FOCAL_GAMMA,
        alpha=FOCAL_ALPHA,
        label_smoothing=LABEL_SMOOTHING,
        num_classes=NUM_CLASSES,
    )

    optimizer = torch.optim.AdamW(
        model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY
    )
    total_steps = len(train_loader) * EPOCHS
    warmup_steps = int(total_steps * WARMUP_RATIO)
    scheduler = get_linear_schedule_with_warmup(
        optimizer,
        num_warmup_steps=warmup_steps,
        num_training_steps=total_steps,
    )
    scaler = GradScaler()

    best_f1 = 0.0
    best_state = None
    patience = 3
    patience_counter = 0

    for epoch in range(EPOCHS):
        train_loss = train_epoch(
            model, train_loader, criterion, optimizer, scheduler, scaler
        )
        val_probs, val_labels_arr, val_f1 = evaluate(model, val_loader)
        print(
            f'  Epoch {epoch + 1}/{EPOCHS}: '
            f'loss={train_loss:.4f}, val_f1={val_f1:.5f}'
        )

        if val_f1 > best_f1:
            best_f1 = val_f1
            best_state = {
                k: v.cpu().clone() for k, v in model.state_dict().items()
            }
            patience_counter = 0
            print(f'    -> Novo melhor F1: {best_f1:.5f}')
        else:
            patience_counter += 1
            if patience_counter >= patience:
                print(f'    Early stopping na epoch {epoch + 1}')
                break

    # Carregar melhor estado do fold
    model.load_state_dict(best_state)
    model.to(DEVICE)

    # OOF predictions
    val_probs_best, _, val_f1_best = evaluate(model, val_loader)
    oof_probs[val_idx] = val_probs_best
    oof_labels[val_idx] = labels[val_idx]
    fold_f1s.append(val_f1_best)

    # Test predictions
    test_probs_fold, _, _ = evaluate(model, test_loader)
    test_probs += test_probs_fold / N_FOLDS

    # Salvar pesos do fold
    torch.save(best_state, f'/kaggle/working/fold_{fold}.pt')
    print(f'  Fold {fold + 1} F1: {val_f1_best:.5f}')

    del model, optimizer, scheduler, scaler, best_state
    gc.collect()
    torch.cuda.empty_cache()

# Resumo
print(f'\n{"="*60}')
print(f'CV F1-macro: {np.mean(fold_f1s):.5f} (+/- {np.std(fold_f1s):.5f})')
print(f'OOF F1-macro (argmax): {compute_f1(oof_probs, oof_labels):.5f}')
for i, f in enumerate(fold_f1s):
    print(f'  Fold {i + 1}: {f:.5f}')

In [ ]:
# ==========================================================================
# OPTUNA CALIBRATION ON OOF
# ==========================================================================
print('Otimizando temperatura + thresholds com Optuna...')
best_temp, best_thresholds, best_oof_f1 = optuna_joint_optimization(
    oof_probs, oof_labels, n_trials=N_TRIALS
)

# Comparar com baseline
baseline_f1 = compute_f1(oof_probs, oof_labels)
print(f'\nBaseline OOF F1 (argmax): {baseline_f1:.5f}')
print(f'Calibrated OOF F1:        {best_oof_f1:.5f}')
print(f'Ganho: +{(best_oof_f1 - baseline_f1) * 100:.3f} pp')

In [ ]:
# ==========================================================================
# SUBMISSION
# ==========================================================================
calibrated_probs = temperature_scale(test_probs, best_temp)
predictions = apply_thresholds(calibrated_probs, best_thresholds)

submission = pd.DataFrame({
    'ID': test_df['ID'],
    'target': predictions,
})
submission.to_csv('/kaggle/working/submission.csv', index=False)

print('submission.csv salvo.')
print(f'Amostras: {len(submission)}')
print(f'\nDistribuicao das predicoes:')
print(submission['target'].value_counts().sort_index())
print(f'\n{submission.to_string(index=False)}')